# 03 — Gold: modelo dimensional e contexto para IA

O fato tem **um registro por filme lançado**. As dimensões descrevem os filmes e as entidades; as tabelas-ponte representam as relações muitos-para-muitos. Os gêneros, atores ou produtoras não estão juntos diretamente no fato: isso repetiria as métricas financeiras.

As chaves substitutas são BIGINT geradas por `row_number()` sobre uma ordenação determinística. Essas chaves não devem ser tratadas como estáveis entre snapshots; em produção incremental, seria necessário um cadastro persistente de chaves.

A dimensão de pessoas tem grão **nome + tipo de atuação**: uma pessoa pode ter uma linha como Ator e outra como Diretor, conforme as colunas pedidas. Avaliações são agregadas em uma linha por filme; notas NULL não entram na média, mas a avaliação permanece na contagem.


In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import re

hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
def parametro(nome, padrao):
    try:
        return dbutils.widgets.get(nome)
    except Exception:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)

CATALOGO = parametro("catalogo", "workspace")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOGO):
    raise ValueError("Use um nome simples de catálogo, com letras, números e underscore.")
DATA_INICIO = datetime.strptime(parametro("data_inicio", (hoje-timedelta(days=6)).strftime("%m-%d-%Y")), "%m-%d-%Y").date()
DATA_FIM = datetime.strptime(parametro("data_fim", hoje.strftime("%m-%d-%Y")), "%m-%d-%Y").date()
if DATA_INICIO > DATA_FIM or DATA_FIM > hoje:
    raise ValueError("O período deve estar em ordem e não pode terminar no futuro.")

def tabela(camada, nome):
    return f"{CATALOGO}.{camada}.{nome}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.gold")
def gravar_gold(df, nome):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(tabela("gold", nome))
def sk(df, chaves, nome):
    janela = Window.orderBy(*[F.col(c).asc() for c in chaves])
    return df.withColumn(nome, F.row_number().over(janela).cast("bigint"))

info = spark.table(tabela("silver", "tb_info_filmes"))
generos = spark.table(tabela("silver", "tb_generos"))
entidades = spark.table(tabela("silver", "tb_pessoas_empresas"))
reviews = spark.table(tabela("silver", "tb_avaliacoes_usuarios"))
financeiro = spark.table(tabela("silver", "tb_financeiro_filmes"))
metricas = spark.table(tabela("silver", "tb_metricas_engajamento"))

dim_movies = sk(info.select("id_filme", "titulo", "data_lancamento", "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme", "sinopse"), ["id_filme"], "sk_movie_id")
gravar_gold(dim_movies.select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme", "sinopse"), "dim_movies")
dim_movies = spark.table(tabela("gold", "dim_movies"))

ids = dim_movies.select("id_filme")
generos = generos.join(ids, "id_filme", "left_semi")
entidades = entidades.join(ids, "id_filme", "left_semi")

dim_genres = sk(generos.select("nome_genero").distinct(), ["nome_genero"], "sk_genre_id")
gravar_gold(dim_genres.select("sk_genre_id", "nome_genero"), "dim_genres")

pessoas = entidades.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
dim_people = sk(pessoas.select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa")).distinct(), ["nome_pessoa", "tipo_pessoa"], "sk_person_id")
gravar_gold(dim_people.select("sk_person_id", "nome_pessoa", "tipo_pessoa"), "dim_people")

produtoras = entidades.filter(F.col("tipo_entidade") == "Produtora")
dim_companies = sk(produtoras.select(F.col("nome_entidade").alias("nome_produtora")).distinct(), ["nome_produtora"], "sk_company_id")
gravar_gold(dim_companies.select("sk_company_id", "nome_produtora"), "dim_companies")

## Pontes e avaliações resumidas

Cada ponte é deduplicada pelo par de chaves estrangeiras. Para a contagem de avaliações, usamos `count(*)`; a média considera somente notas válidas com `avg`. Filmes sem avaliações recebem quantidade zero e média NULL.


In [ ]:
movie_keys = dim_movies.select("id_filme", "sk_movie_id")
dim_genres = spark.table(tabela("gold", "dim_genres"))
dim_people = spark.table(tabela("gold", "dim_people"))
dim_companies = spark.table(tabela("gold", "dim_companies"))

bridge_genre = (generos.join(movie_keys, "id_filme", "inner").join(dim_genres, "nome_genero", "inner")
    .select("sk_movie_id", "sk_genre_id").distinct())
gravar_gold(bridge_genre, "bridge_movie_genre")

bridge_person = (pessoas.select("id_filme", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .join(movie_keys, "id_filme", "inner").join(dim_people, ["nome_pessoa", "tipo_pessoa"], "inner")
    .select("sk_movie_id", "sk_person_id").distinct())
gravar_gold(bridge_person, "bridge_movie_person")

bridge_company = (produtoras.select("id_filme", F.col("nome_entidade").alias("nome_produtora"))
    .join(movie_keys, "id_filme", "inner").join(dim_companies, "nome_produtora", "inner")
    .select("sk_movie_id", "sk_company_id").distinct())
gravar_gold(bridge_company, "bridge_movie_company")

resumo_reviews = (reviews.groupBy("id_filme").agg(
    F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
    F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")))
dim_reviews = (movie_keys.join(resumo_reviews, "id_filme", "left")
    .fillna({"qtd_avaliacoes_usuarios": 0}).drop("id_filme"))
dim_reviews = sk(dim_reviews, ["sk_movie_id"], "sk_review_id")
gravar_gold(dim_reviews.select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"), "dim_reviews")


## Fato de desempenho

É filtrado `status_filme = Lançado` e data de lançamento **até hoje**, evitando considerar uma data futura como lançamento realizado. Uma data de lançamento ausente não permite confirmar que a estreia já ocorreu e não entra no fato. A dimensão mantém os metadados desses filmes.

Joins à esquerda preservam filmes lançados mesmo se faltar financeiro ou engajamento. As duas tabelas Silver têm unicidade por ID; portanto, os joins não multiplicam o grão.


In [ ]:
lancados = dim_movies.filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.lit(hoje)))
financeiro_fato = financeiro.select("id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl")
metricas_fato = metricas.select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
fato = (lancados.select("sk_movie_id", "id_filme")
    .join(financeiro_fato, "id_filme", "left")
    .join(metricas_fato, "id_filme", "left")
    .select("sk_movie_id", "orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl",
        "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"))
gravar_gold(fato, "fact_movies_performance")
display(fato.limit(10))


## Contexto para IA

Para isso, é criado `gold.tb_contexto_ia` com **uma linha por filme**, reunindo metadados, listas deduplicadas de gêneros/pessoas/produtoras e resumo das avaliações. Antes de juntar ao filme, cada relação é agregada por ID para manter o grão. Esta tabela não treina um modelo nem gera respostas automaticamente; prepara o contexto para consumo posterior.


In [ ]:
listas_genero = generos.groupBy("id_filme").agg(F.sort_array(F.collect_set("nome_genero")).alias("generos"))
listas_pessoas = pessoas.groupBy("id_filme").agg(
    F.sort_array(F.collect_set(F.struct(F.col("nome_entidade").alias("nome"), F.col("tipo_entidade").alias("tipo")))).alias("pessoas"))
listas_produtoras = produtoras.groupBy("id_filme").agg(F.sort_array(F.collect_set("nome_entidade")).alias("produtoras"))
contexto = (dim_movies.join(listas_genero, "id_filme", "left")
    .join(listas_pessoas, "id_filme", "left").join(listas_produtoras, "id_filme", "left")
    .join(resumo_reviews, "id_filme", "left").fillna({"qtd_avaliacoes_usuarios": 0}))
gravar_gold(contexto, "tb_contexto_ia")
display(contexto.limit(5))

fato = spark.table(tabela("gold", "fact_movies_performance"))
assert fato.groupBy("sk_movie_id").count().filter("count > 1").limit(1).count() == 0
assert fato.count() == lancados.count()

for nome, chave in [("dim_movies", "sk_movie_id"), ("dim_genres", "sk_genre_id"), ("dim_people", "sk_person_id"), ("dim_companies", "sk_company_id"), ("dim_reviews", "sk_review_id")]:
    df = spark.table(tabela("gold", nome))
    assert df.filter(F.col(chave).isNull()).limit(1).count() == 0
    assert df.groupBy(chave).count().filter("count > 1").limit(1).count() == 0

for ponte, chave, dimensao in [("bridge_movie_genre", "sk_genre_id", "dim_genres"), ("bridge_movie_person", "sk_person_id", "dim_people"), ("bridge_movie_company", "sk_company_id", "dim_companies")]:
    df = spark.table(tabela("gold", ponte))
    assert df.groupBy("sk_movie_id", chave).count().filter("count > 1").limit(1).count() == 0
    assert df.join(dim_movies.select("sk_movie_id"), "sk_movie_id", "left_anti").limit(1).count() == 0
    assert df.join(spark.table(tabela("gold", dimensao)).select(chave), chave, "left_anti").limit(1).count() == 0
    
assert spark.table(tabela("gold", "dim_reviews")).join(dim_movies.select("sk_movie_id"), "sk_movie_id", "left_anti").limit(1).count() == 0
print("PKs, FKs, pontes e grão do fato validados.")